## SRP094638

**paper:** [PMID: 28843847](https://www.sciencedirect.com/science/article/pii/S1874778717301113?via%3Dihub) - Skin Transcriptomes of common bottlenose dolphins (Tursiops truncatus) from the northern Gulf of Mexico and southeastern U.S. Atlantic coasts, 2017

**date, curator:** 2026-04-10, Sara Carsanaro

**notes**
* some blubber was included in the sample, so annotating as integument is better to include the hypodermis
* all the animals were found in the wild so they are definitely fully formed stage

### annotation summary

In [22]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,skin,UBERON:0002199,integument,perfect match


In [23]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,NA,UBERON:0000066,fully formed stage,perfect match


### set variables, import packages, define functions

In [2]:
experiment_id = "SRP094638"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [3]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [4]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
0it [00:00, ?it/s]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes


### library annnotations

In [5]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX2398836,SRP094638,Illumina HiSeq 2500,SRS1839755,,,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2417918,skin,NA,,,,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,TYP140815-02,"SAMN06113344,GSM2417918",,,,,,,,,,,10/04/2026,"Qiagen RNeasy Lipid Tissue Mini Kit with on-column DNase digestion. NEBNext Ultra Directional RNA Library Prep Kit for Illumina, indexed with the NEBNextMulitplex Oligos for Illumina.",,,,skin,,,,TRANSCRIPTOMIC,cDNA
1,SRX2398835,SRP094638,Illumina HiSeq 2500,SRS1839754,,,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2417917,skin,NA,,,,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,TYP140808-05,"SAMN06113345,GSM2417917",,,,,,,,,,,10/04/2026,"Qiagen RNeasy Lipid Tissue Mini Kit with on-column DNase digestion. NEBNext Ultra Directional RNA Library Prep Kit for Illumina, indexed with the NEBNextMulitplex Oligos for Illumina.",,,,skin,,,,TRANSCRIPTOMIC,cDNA
2,SRX2398834,SRP094638,Illumina HiSeq 2500,SRS1839756,,,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2417916,skin,NA,,,,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,TYP140808-01,"SAMN06113346,GSM2417916",,,,,,,,,,,10/04/2026,"Qiagen RNeasy Lipid Tissue Mini Kit with on-column DNase digestion. NEBNext Ultra Directional RNA Library Prep Kit for Illumina, indexed with the NEBNextMulitplex Oligos for Illumina.",,,,skin,,,,TRANSCRIPTOMIC,cDNA
3,SRX2398833,SRP094638,Illumina HiSeq 2500,SRS1839753,,,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2417915,skin,NA,,,,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,TYP140807-01,"SAMN06113347,GSM2417915",,,,,,,,,,,10/04/2026,"Qiagen RNeasy Lipid Tissue Mini Kit with on-column DNase digestion. NEBNext Ultra Directional RNA Library Prep Kit for Illumina, indexed with the NEBNextMulitplex Oligos for Illumina.",,,,skin,,,,TRANSCRIPTOMIC,cDNA
4,SRX2398832,SRP094638,Illumina HiSeq 2500,SRS1839752,,,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2417914,skin,NA,,,,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,TYP140806-01,"SAMN06113348,GSM2417914",,,,,,,,,,,10/04/2026,"Qiagen RNeasy Lipid Tissue Mini Kit with on-column DNase digestion. NEBNext Ultra Directional RNA Library Prep Kit for Illumina, indexed with the NEBNextMulitplex Oligos for Illumina.",,,,skin,,,,TRANSCRIPTOMIC,cDNA
5,SRX2398831,SRP094638,Illumina HiSeq 2500,SRS1839751,,,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2417913,skin,NA,,,,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,TYP140801-04,"SAMN06113349,GSM2417913",,,,,,,,,,,10/04/2026,"Qiagen RNeasy Lipid Tissue Mini Kit with on-column DNase digestion. NEBNext Ultra Directional RNA Library Prep Kit for Illumina, indexed with the NEBNextMulitplex Oligos for Illumina.",,,,skin,,,,TRANSCRIPTOMIC,cDNA
6,SRX2398830,SRP094638,Illumina HiSeq 2500,SRS1839750,,,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2417912,skin,NA,,,,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,TYP140730-05,"SAMN06113350,GSM2417912",,,,,,,,,,,10/04/2026,"Qiagen RNeasy Lipid Tissue Mini Kit with on-column DNase digestion. NEBNext Ultra Directional RNA Library Prep Kit for Illumina, indexed with the NEBNextMulitplex Oligos for Illumina.",,,,skin,,,,TRANSCRIPTOMIC,cDNA
7,SRX2398829,SRP094638,Illumina HiSeq 2500,SRS1839749,,,,,http://www.ncbi.nlm.nih.

#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [6]:
unique_sorted(library, "infoOrgan")

['skin']


In [7]:

# all
library.loc[:,'anatId'] = 'UBERON:0002199'
library.loc[:,'anatName'] = 'integument'
# perfect match, missing child term, other
library.loc[:,'anatAnnotationStatus'] = 'perfect match'
# partial sampling, full sampling, not documented
library.loc[:,'anatBiologicalStatus'] = 'not documented'



# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX2398836,SRP094638,Illumina HiSeq 2500,SRS1839755,UBERON:0002199,integument,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2417918,skin,NA,perfect match,not documented,,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,TYP140815-02,"SAMN06113344,GSM2417918",,,,,,,,,,,10/04/2026,"Qiagen RNeasy Lipid Tissue Mini Kit with on-column DNase digestion. NEBNext Ultra Directional RNA Library Prep Kit for Illumina, indexed with the NEBNextMulitplex Oligos for Illumina.",,,,skin,,,,TRANSCRIPTOMIC,cDNA
1,SRX2398835,SRP094638,Illumina HiSeq 2500,SRS1839754,UBERON:0002199,integument,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2417917,skin,NA,perfect match,not documented,,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,TYP140808-05,"SAMN06113345,GSM2417917",,,,,,,,,,,10/04/2026,"Qiagen RNeasy Lipid Tissue Mini Kit with on-column DNase digestion. NEBNext Ultra Directional RNA Library Prep Kit for Illumina, indexed with the NEBNextMulitplex Oligos for Illumina.",,,,skin,,,,TRANSCRIPTOMIC,cDNA
2,SRX2398834,SRP094638,Illumina HiSeq 2500,SRS1839756,UBERON:0002199,integument,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2417916,skin,NA,perfect match,not documented,,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,TYP140808-01,"SAMN06113346,GSM2417916",,,,,,,,,,,10/04/2026,"Qiagen RNeasy Lipid Tissue Mini Kit with on-column DNase digestion. NEBNext Ultra Directional RNA Library Prep Kit for Illumina, indexed with the NEBNextMulitplex Oligos for Illumina.",,,,skin,,,,TRANSCRIPTOMIC,cDNA
3,SRX2398833,SRP094638,Illumina HiSeq 2500,SRS1839753,UBERON:0002199,integument,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2417915,skin,NA,perfect match,not documented,,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,TYP140807-01,"SAMN06113347,GSM2417915",,,,,,,,,,,10/04/2026,"Qiagen RNeasy Lipid Tissue Mini Kit with on-column DNase digestion. NEBNext Ultra Directional RNA Library Prep Kit for Illumina, indexed with the NEBNextMulitplex Oligos for Illumina.",,,,skin,,,,TRANSCRIPTOMIC,cDNA
4,SRX2398832,SRP094638,Illumina HiSeq 2500,SRS1839752,UBERON:0002199,integument,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2417914,skin,NA,perfect match,not documented,,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,TYP140806-01,"SAMN06113348,GSM2417914",,,,,,,,,,,10/04/2026,"Qiagen RNeasy Lipid Tissue Mini Kit with on-column DNase digestion. NEBNext Ultra Directional RNA Library Prep Kit for Illumina, indexed with the NEBNextMulitplex Oligos for Illumina.",,,,skin,,,,TRANSCRIPTOMIC,cDNA
5,SRX2398831,SRP094638,Illumina HiSeq 2500,SRS1839751,UBERON:0002199,integument,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2417913,skin,NA,perfect match,not documented,,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,TYP140801-04,"SAMN06113349,GSM2417913",,,,,,,,,,,10/04/2026,"Qiagen RNeasy Lipid Tissue Mini Kit with on-column DNase digestion. NEBNext Ultra Directional RNA Library Prep Kit for Illumina, indexed with the NEBNextMulitplex Oligos for Illumina.",,,,skin,,,,TRANSCRIPTOMIC,cDNA
6,SRX2398830,SRP094638,Illumina HiSeq 2500,SRS1839750,UBERON:0002199,integument,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2417912,skin,NA,perfect match,not documented,,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,TY

#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [8]:
unique_sorted(library, "infoStage")

['NA']


In [9]:
# all
library.loc[:,'stageId'] = 'UBERON:0000066'
library.loc[:,'stageName'] = 'fully formed stage'
# perfect match, missing child term, other
library.loc[:,'stageAnnotationStatus'] = 'perfect match'


# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX2398836,SRP094638,Illumina HiSeq 2500,SRS1839755,UBERON:0002199,integument,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2417918,skin,NA,perfect match,not documented,perfect match,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,TYP140815-02,"SAMN06113344,GSM2417918",,,,,,,,,,,10/04/2026,"Qiagen RNeasy Lipid Tissue Mini Kit with on-column DNase digestion. NEBNext Ultra Directional RNA Library Prep Kit for Illumina, indexed with the NEBNextMulitplex Oligos for Illumina.",,,,skin,,,,TRANSCRIPTOMIC,cDNA
1,SRX2398835,SRP094638,Illumina HiSeq 2500,SRS1839754,UBERON:0002199,integument,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2417917,skin,NA,perfect match,not documented,perfect match,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,TYP140808-05,"SAMN06113345,GSM2417917",,,,,,,,,,,10/04/2026,"Qiagen RNeasy Lipid Tissue Mini Kit with on-column DNase digestion. NEBNext Ultra Directional RNA Library Prep Kit for Illumina, indexed with the NEBNextMulitplex Oligos for Illumina.",,,,skin,,,,TRANSCRIPTOMIC,cDNA
2,SRX2398834,SRP094638,Illumina HiSeq 2500,SRS1839756,UBERON:0002199,integument,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2417916,skin,NA,perfect match,not documented,perfect match,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,TYP140808-01,"SAMN06113346,GSM2417916",,,,,,,,,,,10/04/2026,"Qiagen RNeasy Lipid Tissue Mini Kit with on-column DNase digestion. NEBNext Ultra Directional RNA Library Prep Kit for Illumina, indexed with the NEBNextMulitplex Oligos for Illumina.",,,,skin,,,,TRANSCRIPTOMIC,cDNA
3,SRX2398833,SRP094638,Illumina HiSeq 2500,SRS1839753,UBERON:0002199,integument,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2417915,skin,NA,perfect match,not documented,perfect match,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,TYP140807-01,"SAMN06113347,GSM2417915",,,,,,,,,,,10/04/2026,"Qiagen RNeasy Lipid Tissue Mini Kit with on-column DNase digestion. NEBNext Ultra Directional RNA Library Prep Kit for Illumina, indexed with the NEBNextMulitplex Oligos for Illumina.",,,,skin,,,,TRANSCRIPTOMIC,cDNA
4,SRX2398832,SRP094638,Illumina HiSeq 2500,SRS1839752,UBERON:0002199,integument,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2417914,skin,NA,perfect match,not documented,perfect match,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,TYP140806-01,"SAMN06113348,GSM2417914",,,,,,,,,,,10/04/2026,"Qiagen RNeasy Lipid Tissue Mini Kit with on-column DNase digestion. NEBNext Ultra Directional RNA Library Prep Kit for Illumina, indexed with the NEBNextMulitplex Oligos for Illumina.",,,,skin,,,,TRANSCRIPTOMIC,cDNA
5,SRX2398831,SRP094638,Illumina HiSeq 2500,SRS1839751,UBERON:0002199,integument,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2417913,skin,NA,perfect match,not documented,perfect match,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,TYP140801-04,"SAMN06113349,GSM2417913",,,,,,,,,,,10/04/2026,"Qiagen RNeasy Lipid Tissue Mini Kit with on-column DNase digestion. NEBNext Ultra Directional RNA Library Prep Kit for Illumina, indexed with the NEBNextMulitplex Oligos for Illumina.",,,,skin,,,,TRANSCR

#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [ ]:
#library.loc[:,'sex'] = ''
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [10]:
# making these variables because we use them again in the experiment file
my_protocol = 'NEBNext Ultra Directional RNA Library Prep Kit'
# full_length or 3'
my_protocolType = 'full_length'

library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX2398836,SRP094638,Illumina HiSeq 2500,SRS1839755,UBERON:0002199,integument,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2417918,skin,NA,perfect match,not documented,perfect match,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,TYP140815-02,"SAMN06113344,GSM2417918",,,,,,,,,,,10/04/2026,"Qiagen RNeasy Lipid Tissue Mini Kit with on-column DNase digestion. NEBNext Ultra Directional RNA Library Prep Kit for Illumina, indexed with the NEBNextMulitplex Oligos for Illumina.",,,,skin,,,,TRANSCRIPTOMIC,cDNA
1,SRX2398835,SRP094638,Illumina HiSeq 2500,SRS1839754,UBERON:0002199,integument,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2417917,skin,NA,perfect match,not documented,perfect match,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,TYP140808-05,"SAMN06113345,GSM2417917",,,,,,,,,,,10/04/2026,"Qiagen RNeasy Lipid Tissue Mini Kit with on-column DNase digestion. NEBNext Ultra Directional RNA Library Prep Kit for Illumina, indexed with the NEBNextMulitplex Oligos for Illumina.",,,,skin,,,,TRANSCRIPTOMIC,cDNA
2,SRX2398834,SRP094638,Illumina HiSeq 2500,SRS1839756,UBERON:0002199,integument,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2417916,skin,NA,perfect match,not documented,perfect match,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,TYP140808-01,"SAMN06113346,GSM2417916",,,,,,,,,,,10/04/2026,"Qiagen RNeasy Lipid Tissue Mini Kit with on-column DNase digestion. NEBNext Ultra Directional RNA Library Prep Kit for Illumina, indexed with the NEBNextMulitplex Oligos for Illumina.",,,,skin,,,,TRANSCRIPTOMIC,cDNA
3,SRX2398833,SRP094638,Illumina HiSeq 2500,SRS1839753,UBERON:0002199,integument,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2417915,skin,NA,perfect match,not documented,perfect match,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,TYP140807-01,"SAMN06113347,GSM2417915",,,,,,,,,,,10/04/2026,"Qiagen RNeasy Lipid Tissue Mini Kit with on-column DNase digestion. NEBNext Ultra Directional RNA Library Prep Kit for Illumina, indexed with the NEBNextMulitplex Oligos for Illumina.",,,,skin,,,,TRANSCRIPTOMIC,cDNA
4,SRX2398832,SRP094638,Illumina HiSeq 2500,SRS1839752,UBERON:0002199,integument,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2417914,skin,NA,perfect match,not documented,perfect match,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,TYP140806-01,"SAMN06113348,GSM2417914",,,,,,,,,,,10/04/2026,"Qiagen RNeasy Lipid Tissue Mini Kit with on-column DNase digestion. NEBNext Ultra Directional RNA Library Prep Kit for Illumina, indexed with the NEBNextMulitplex Oligos for Illumina.",,,,skin,,,,TRANSCRIPTOMIC,cDNA
5,SRX2398831,SRP094638,Illumina HiSeq 2500,SRS1839751,UBERON:0002199,integument,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2417913,skin,NA,perfect match,not documented,perfect match,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,TYP140801-04,"SAMN06113349,GSM2417913",,,,,,,,,,,10/04/2026,"Qiagen RNeasy Lipid Tissue Mini Kit with on-column DNase digestion. NEBNext Ultra Directional RNA Library Prep Kit for Illumina, indexed with the NEBNextMulitplex Oligos for Illumina.",,,,ski

#### globin, replicates

In [11]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [12]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-04-10'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX2398836,SRP094638,Illumina HiSeq 2500,SRS1839755,UBERON:0002199,integument,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2417918,skin,NA,perfect match,not documented,perfect match,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,TYP140815-02,"SAMN06113344,GSM2417918",,,,,,,,,,SAC,2026-04-10,"Qiagen RNeasy Lipid Tissue Mini Kit with on-column DNase digestion. NEBNext Ultra Directional RNA Library Prep Kit for Illumina, indexed with the NEBNextMulitplex Oligos for Illumina.",,,,skin,,,,TRANSCRIPTOMIC,cDNA
1,SRX2398835,SRP094638,Illumina HiSeq 2500,SRS1839754,UBERON:0002199,integument,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2417917,skin,NA,perfect match,not documented,perfect match,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,TYP140808-05,"SAMN06113345,GSM2417917",,,,,,,,,,SAC,2026-04-10,"Qiagen RNeasy Lipid Tissue Mini Kit with on-column DNase digestion. NEBNext Ultra Directional RNA Library Prep Kit for Illumina, indexed with the NEBNextMulitplex Oligos for Illumina.",,,,skin,,,,TRANSCRIPTOMIC,cDNA
2,SRX2398834,SRP094638,Illumina HiSeq 2500,SRS1839756,UBERON:0002199,integument,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2417916,skin,NA,perfect match,not documented,perfect match,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,TYP140808-01,"SAMN06113346,GSM2417916",,,,,,,,,,SAC,2026-04-10,"Qiagen RNeasy Lipid Tissue Mini Kit with on-column DNase digestion. NEBNext Ultra Directional RNA Library Prep Kit for Illumina, indexed with the NEBNextMulitplex Oligos for Illumina.",,,,skin,,,,TRANSCRIPTOMIC,cDNA
3,SRX2398833,SRP094638,Illumina HiSeq 2500,SRS1839753,UBERON:0002199,integument,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2417915,skin,NA,perfect match,not documented,perfect match,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,TYP140807-01,"SAMN06113347,GSM2417915",,,,,,,,,,SAC,2026-04-10,"Qiagen RNeasy Lipid Tissue Mini Kit with on-column DNase digestion. NEBNext Ultra Directional RNA Library Prep Kit for Illumina, indexed with the NEBNextMulitplex Oligos for Illumina.",,,,skin,,,,TRANSCRIPTOMIC,cDNA
4,SRX2398832,SRP094638,Illumina HiSeq 2500,SRS1839752,UBERON:0002199,integument,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2417914,skin,NA,perfect match,not documented,perfect match,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,TYP140806-01,"SAMN06113348,GSM2417914",,,,,,,,,,SAC,2026-04-10,"Qiagen RNeasy Lipid Tissue Mini Kit with on-column DNase digestion. NEBNext Ultra Directional RNA Library Prep Kit for Illumina, indexed with the NEBNextMulitplex Oligos for Illumina.",,,,skin,,,,TRANSCRIPTOMIC,cDNA
5,SRX2398831,SRP094638,Illumina HiSeq 2500,SRS1839751,UBERON:0002199,integument,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2417913,skin,NA,perfect match,not documented,perfect match,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,TYP140801-04,"SAMN06113349,GSM2417913",,,,,,,,,,SAC,2026-04-10,"Qiagen RNeasy Lipid Tissue Mini Kit with on-column DNase digestion. NEBNext Ultra Directional RNA Library Prep Kit for Illumina, indexed with the NEBNextMulitplex Oligos for

#### comments

In [13]:
library.loc[:,'comment'] = 'PMID: 28843847, sample is skin plus some blubber so integument is appropriate here'

#### save complete file with correct columns

In [14]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX2398836,SRP094638,Illumina HiSeq 2500,SRS1839755,UBERON:0002199,integument,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2417918,skin,NA,perfect match,not documented,perfect match,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,TYP140815-02,"SAMN06113344,GSM2417918",,,,,,,"PMID: 28843847, sample is skin plus some blubber so integument is appropriate here",,,SAC,2026-04-10
1,SRX2398835,SRP094638,Illumina HiSeq 2500,SRS1839754,UBERON:0002199,integument,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2417917,skin,NA,perfect match,not documented,perfect match,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,TYP140808-05,"SAMN06113345,GSM2417917",,,,,,,"PMID: 28843847, sample is skin plus some blubber so integument is appropriate here",,,SAC,2026-04-10
2,SRX2398834,SRP094638,Illumina HiSeq 2500,SRS1839756,UBERON:0002199,integument,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2417916,skin,NA,perfect match,not documented,perfect match,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,TYP140808-01,"SAMN06113346,GSM2417916",,,,,,,"PMID: 28843847, sample is skin plus some blubber so integument is appropriate here",,,SAC,2026-04-10
3,SRX2398833,SRP094638,Illumina HiSeq 2500,SRS1839753,UBERON:0002199,integument,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2417915,skin,NA,perfect match,not documented,perfect match,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,TYP140807-01,"SAMN06113347,GSM2417915",,,,,,,"PMID: 28843847, sample is skin plus some blubber so integument is appropriate here",,,SAC,2026-04-10
4,SRX2398832,SRP094638,Illumina HiSeq 2500,SRS1839752,UBERON:0002199,integument,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2417914,skin,NA,perfect match,not documented,perfect match,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,TYP140806-01,"SAMN06113348,GSM2417914",,,,,,,"PMID: 28843847, sample is skin plus some blubber so integument is appropriate here",,,SAC,2026-04-10
5,SRX2398831,SRP094638,Illumina HiSeq 2500,SRS1839751,UBERON:0002199,integument,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2417913,skin,NA,perfect match,not documented,perfect match,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,TYP140801-04,"SAMN06113349,GSM2417913",,,,,,,"PMID: 28843847, sample is skin plus some blubber so integument is appropriate here",,,SAC,2026-04-10
6,SRX2398830,SRP094638,Illumina HiSeq 2500,SRS1839750,UBERON:0002199,integument,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2417912,skin,NA,perfect match,not documented,perfect match,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,TYP140730-05,"SAMN06113350,GSM2417912",,,,,,,"PMID: 28843847, sample is skin plus some blubber so integument is appropriate here",,,SAC,2026-04-10
7,SRX2398829,SRP094638,Illumina HiSeq 2500,SRS1839749,UBERON:0002199,integument,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2417911,skin,NA,perfect match,not documented,perfect match,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,TYP140717-03,"SAMN06113351,GSM2417911",,,,,,,"PMID: 28843847, sample is skin plus some blubber so integument is appropriate here",,,SAC,2026-04-1

### experiment annotations

In [15]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP094638,Skin Transcriptomics of Common Bottlenose Dolphins (Tursiops truncatus) from the Northern Gulf of Mexico and Southeastern U.S. Atlantic Coasts,"Common bottlenose dolphins serve as sentinels for the health of their coastal environments as they are susceptible to health impacts from anthropogenic inputs through both direct exposure and food web magnification. Remote biopsy samples have been widely used to reveal contaminant burdens in free-ranging bottlenose dolphins, but do not address the health consequences of this exposure. To gain insight into whether remote biopsies can also identify health impacts associated with contaminant burdens, we employed RNA sequencing (RNA-seq) to interrogate the transcriptomes of remote skin biopsies from 116 bottlenose dolphins from the northern Gulf of Mexico and southeastern U.S. Atlantic coasts. Gene expression was analyzed using principal component analysis, differential expression testing, and gene co-expression networks, and the results correlated to season, location, and contaminant burden. Season had a significant impact, with over 30% of genes differentially expressed between spring/summer and winter months. Geographic location exhibited lesser effects on the transcriptome, with 15% of genes differentially expressed between the northern Gulf of Mexico and the southeastern U.S. Atlantic locations. Despite a large overlap between the seasonal and geographical gene sets, the pathways altered in the observed gene expression profiles were somewhat distinct. Co-regulated gene modules and differential expression analysis both identified epidermal development and cellular architecture pathways to be expressed at lower levels in animals from the northern Gulf of Mexico. Although contaminant burdens measured were not significantly different between regions, some correlation with contaminant loads in individuals was observed among co-expressed gene modules, but these did not include classical detoxification pathways. Instead, this study identified other, possibly downstream pathways, including those involved in cellular architecture, immune response, and oxidative stress, that may prove to be contaminant responsive markers in bottlenose dolphin skin. Overall design: RNA-seq of skin from 116 common bottlenose dolphins.",SRA,,,,NEBNext Ultra Directional RNA Library Prep Kit,full_length,GSE90941,PRJNA356411,28843847,,10.1016/j.margen.2017.08.002,,


#### experiment and protocol details

In [16]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

116

In [17]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'total'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP094638,Skin Transcriptomics of Common Bottlenose Dolphins (Tursiops truncatus) from the Northern Gulf of Mexico and Southeastern U.S. Atlantic Coasts,"Common bottlenose dolphins serve as sentinels for the health of their coastal environments as they are susceptible to health impacts from anthropogenic inputs through both direct exposure and food web magnification. Remote biopsy samples have been widely used to reveal contaminant burdens in free-ranging bottlenose dolphins, but do not address the health consequences of this exposure. To gain insight into whether remote biopsies can also identify health impacts associated with contaminant burdens, we employed RNA sequencing (RNA-seq) to interrogate the transcriptomes of remote skin biopsies from 116 bottlenose dolphins from the northern Gulf of Mexico and southeastern U.S. Atlantic coasts. Gene expression was analyzed using principal component analysis, differential expression testing, and gene co-expression networks, and the results correlated to season, location, and contaminant burden. Season had a significant impact, with over 30% of genes differentially expressed between spring/summer and winter months. Geographic location exhibited lesser effects on the transcriptome, with 15% of genes differentially expressed between the northern Gulf of Mexico and the southeastern U.S. Atlantic locations. Despite a large overlap between the seasonal and geographical gene sets, the pathways altered in the observed gene expression profiles were somewhat distinct. Co-regulated gene modules and differential expression analysis both identified epidermal development and cellular architecture pathways to be expressed at lower levels in animals from the northern Gulf of Mexico. Although contaminant burdens measured were not significantly different between regions, some correlation with contaminant loads in individuals was observed among co-expressed gene modules, but these did not include classical detoxification pathways. Instead, this study identified other, possibly downstream pathways, including those involved in cellular architecture, immune response, and oxidative stress, that may prove to be contaminant responsive markers in bottlenose dolphin skin. Overall design: RNA-seq of skin from 116 common bottlenose dolphins.",SRA,total,Bgee 1K,116,NEBNext Ultra Directional RNA Library Prep Kit,full_length,GSE90941,PRJNA356411,28843847,,10.1016/j.margen.2017.08.002,,


#### paper and xrefs

In [18]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '28843847'
experiment.loc[:,'reference_url'] = 'https://www.sciencedirect.com/science/article/pii/S1874778717301113?via%3Dihub'
experiment.loc[:,'DOI'] = '10.1016/j.margen.2017.08.002'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP094638,Skin Transcriptomics of Common Bottlenose Dolphins (Tursiops truncatus) from the Northern Gulf of Mexico and Southeastern U.S. Atlantic Coasts,"Common bottlenose dolphins serve as sentinels for the health of their coastal environments as they are susceptible to health impacts from anthropogenic inputs through both direct exposure and food web magnification. Remote biopsy samples have been widely used to reveal contaminant burdens in free-ranging bottlenose dolphins, but do not address the health consequences of this exposure. To gain insight into whether remote biopsies can also identify health impacts associated with contaminant burdens, we employed RNA sequencing (RNA-seq) to interrogate the transcriptomes of remote skin biopsies from 116 bottlenose dolphins from the northern Gulf of Mexico and southeastern U.S. Atlantic coasts. Gene expression was analyzed using principal component analysis, differential expression testing, and gene co-expression networks, and the results correlated to season, location, and contaminant burden. Season had a significant impact, with over 30% of genes differentially expressed between spring/summer and winter months. Geographic location exhibited lesser effects on the transcriptome, with 15% of genes differentially expressed between the northern Gulf of Mexico and the southeastern U.S. Atlantic locations. Despite a large overlap between the seasonal and geographical gene sets, the pathways altered in the observed gene expression profiles were somewhat distinct. Co-regulated gene modules and differential expression analysis both identified epidermal development and cellular architecture pathways to be expressed at lower levels in animals from the northern Gulf of Mexico. Although contaminant burdens measured were not significantly different between regions, some correlation with contaminant loads in individuals was observed among co-expressed gene modules, but these did not include classical detoxification pathways. Instead, this study identified other, possibly downstream pathways, including those involved in cellular architecture, immune response, and oxidative stress, that may prove to be contaminant responsive markers in bottlenose dolphin skin. Overall design: RNA-seq of skin from 116 common bottlenose dolphins.",SRA,total,Bgee 1K,116,NEBNext Ultra Directional RNA Library Prep Kit,full_length,GSE90941,PRJNA356411,28843847,https://www.sciencedirect.com/science/article/pii/S1874778717301113?via%3Dihub,10.1016/j.margen.2017.08.002,,


#### comments

In [ ]:
#experiment.loc[:,'comment'] = ''

display_df(experiment)

#### save complete file

In [19]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [20]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [21]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 0
Errors: 0
Warnings: 0
Top codes:


#### check columns match

In [24]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [25]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
61005,DRX497176,DRP010786,Illumina NovaSeq 6000,DRS341452,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Blacky_spring,SAMD00656858,,,,,,,PMID: 38822022,spring,,SAC,2026-04-10
61006,DRX497175,DRP010786,Illumina NovaSeq 6000,DRS341451,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Aria_spring,SAMD00656857,,,,,,,PMID: 38822022,spring,,SAC,2026-04-10
61007,SRX2398836,SRP094638,Illumina HiSeq 2500,SRS1839755,UBERON:0002199,integument,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?...,skin,NA,perfect match,not documented,perfect match,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,TYP140815-02,"SAMN06113344,GSM2417918",,,,,,,"PMID: 28843847, sample is skin plus some blubb...",,,SAC,2026-04-10
61008,SRX2398835,SRP094638,Illumina HiSeq 2500,SRS1839754,UBERON:0002199,integument,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?...,skin,NA,perfect match,not documented,perfect match,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,TYP140808-05,"SAMN06113345,GSM2417917",,,,,,,"PMID: 28843847, sample is skin plus some blubb...",,,SAC,2026-04-10
61009,SRX2398834,SRP094638,Illumina HiSeq 2500,SRS1839756,UBERON:0002199,integument,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?...,skin,NA,perfect match,not documented,perfect match,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,TYP140808-01,"SAMN06113346,GSM2417916",,,,,,,"PMID: 28843847, sample is skin plus some blubb...",,,SAC,2026-04-10
61010,SRX2398833,SRP094638,Illumina HiSeq 2500,SRS1839753,UBERON:0002199,integument,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?...,skin,NA,perfect match,not documented,perfect match,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,TYP140807-01,"SAMN06113347,GSM2417915",,,,,,,"PMID: 28843847, sample is skin plus some blubb...",,,SAC,2026-04-10
61011,SRX2398832,SRP094638,Illumina HiSeq 2500,SRS1839752,UBERON:0002199,integument,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?...,skin,NA,perfect match,not documented,perfect match,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,TYP140806-01,"SAMN06113348,GSM2417914",,,,,,,"PMID: 28843847, sample is skin plus some blubb...",,,SAC,2026-04-10


In [26]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1166,SRP359220,Pseudogenization of Paraoxonase 1 (PON1) in Pi...,This data is part of a larger project interest...,SRA,total,Bgee 1K,4,,,,PRJNA805216,37146172,https://pmc.ncbi.nlm.nih.gov/articles/PMC10202...,10.1093/molbev/msad104,,
1167,DRP010786,Dolphin_blubber_transcriptome,A comprehensive gene expression analysis was c...,SRA,total,Bgee 1K,32,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,,PRJDB16986,38822022,https://pmc.ncbi.nlm.nih.gov/articles/PMC11143...,10.1038/s41598-024-63018-7,,
1168,SRP094638,Skin Transcriptomics of Common Bottlenose Dolp...,Common bottlenose dolphins serve as sentinels ...,SRA,total,Bgee 1K,116,NEBNext Ultra Directional RNA Library Prep Kit,full_length,GSE90941,PRJNA356411,28843847,https://www.sciencedirect.com/science/article/...,10.1016/j.margen.2017.08.002,,


### add annotations to git

In [27]:
! git pull

Already up to date.


In [28]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [ ]:
! git status

In [29]:
! git add $git_experiment_path $git_library_path

In [30]:
! git commit -m $commit_message_exp

[develop d0f16a7] adding annotated bulk experiment SRP094638
 2 files changed, 117 insertions(+)


In [31]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 5.62 KiB | 1.40 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   a189e37..d0f16a7  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push